# 01 Data Check

## 0. Research Questions
1. **Proportion question:** Is the proportion of students who reported feeling sad or hopeless different from 0.30?
2. **Mean question:** Is the mean body weight of students different from 68.0 kg?

----- **Additional eda** -----
1. Is SadOrHopeless more common among students who reported smoking on at least 1 day in the past 30 days, and does this association vary across age groups?
2. Among current smokers, does smoking frequency appear to differ by SadOrHopeless status?

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

#---------------設定整個專案資料夾路徑------------------------
ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_combined_processed.csv"
TAB_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
REF_DIR = ROOT / "references"
SUM_DIR = ROOT / "outputs" / "summary"

for p in [FIG_DIR, TAB_DIR, SUM_DIR, REF_DIR, PROCESSED_PATH.parent]:
    p.mkdir(parents=True, exist_ok=True)
#---------------設定整個專案資料夾路徑------------------------

#---------------正式主分析變數------------------------
BEHAVIOR_VAR = "SadOrHopeless"
BEHAVIOR_P0 = 0.30

CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0
#---------------正式主分析變數------------------------

#---------------延伸探索變數------------------------
SMOKE_VAR = "CurrentCigaretteUse"
AGE_VAR = "HowOldAreYou"
#---------------延伸探索變數------------------------

#---------------整合處理 Raw Data -> Processed Data------------------------
def ensure_processed():
    raw = pd.read_csv(RAW_PATH)

    # ===== 正式 Behavior Variable: SadOrHopeless =====
    behavior_raw = raw[BEHAVIOR_VAR]
    behavior_clean = behavior_raw.where(behavior_raw.isin([1, 2]))

    sad_binary = pd.Series(
        np.where(behavior_clean == 1, 1,
                 np.where(behavior_clean == 2, 0, np.nan)),
        name="sad_binary"
    )

    # ===== 正式 Continuous Variable: Weight =====
    cont_clean = pd.to_numeric(raw[CONT_VAR], errors="coerce")
    cont_clean = cont_clean.where(cont_clean > 0)

    # ===== 延伸探索：CurrentCigaretteUse =====
    smoke_raw = raw[SMOKE_VAR]

    smoker_binary = pd.Series(
        np.where(smoke_raw.isin([2, 3, 4, 5, 6, 7]), 1,
                 np.where(smoke_raw == 1, 0, np.nan)),
        name="smoker_binary"
    )

    freq_group_map = {
        2: "Light (1~5 days)",
        3: "Light (1~5 days)",
        4: "Moderate (6~19 days)",
        5: "Moderate (6~19 days)",
        6: "Frequent (20~30 days)",
        7: "Frequent (20~30 days)"
    }

    smoking_freq_group = pd.Categorical(
        smoke_raw.map(freq_group_map),
        categories=["Light (1~5 days)", "Moderate (6~19 days)", "Frequent (20~30 days)"],
        ordered=True
    )

    smoking_freq_score = pd.Series(
        np.where(smoke_raw.isin([2, 3, 4, 5, 6, 7]), smoke_raw, np.nan),
        name="smoking_freq_score"
    )

    # ===== 延伸探索：HowOldAreYou =====
    age_raw = raw[AGE_VAR]

    age_label_map = {
        1: "<=12",
        2: "13",
        3: "14",
        4: "15",
        5: "16",
        6: "17",
        7: "18+"
    }

    age_group_map = {
        1: "<=14",
        2: "<=14",
        3: "<=14",
        4: "15",
        5: "16",
        6: "17",
        7: "18+"
    }

    age_label = pd.Series(age_raw.map(age_label_map), name="age_label")

    age_group = pd.Series(
        pd.Categorical(
            age_raw.map(age_group_map),
            categories=["<=14", "15", "16", "17", "18+"],
            ordered=True
        ),
        name="age_group"
    )

    # ===== 合併成單一 processed 檔 =====
    processed = pd.DataFrame({
        BEHAVIOR_VAR: behavior_raw,
        "sad_binary": sad_binary,
        CONT_VAR: cont_clean,
        SMOKE_VAR: smoke_raw,
        "smoker_binary": smoker_binary,
        "smoking_freq_group": smoking_freq_group,
        "smoking_freq_score": smoking_freq_score,
        AGE_VAR: age_raw,
        "age_label": age_label,
        "age_group": age_group
    })

    processed.to_csv(PROCESSED_PATH, index=False)
    return raw, processed
#---------------整合處理 Raw Data -> Processed Data------------------------

raw, processed = ensure_processed()

print("Raw data shape:", raw.shape)
print("Processed data shape:", processed.shape)
print("Saved processed dataset to:", PROCESSED_PATH)

Raw data shape: (14041, 103)
Processed data shape: (14041, 10)
Saved processed dataset to: C:\Users\asd45\Downloads\combine\data\processed\yrbs_combined_processed.csv


## 2. Formal: Variable Definition and Coding

### 2.1 Behavior Variable for Proportion Analysis
- **Variable name:** `SadOrHopeless`
- **What the variable measures:** whether the student felt so sad or hopeless almost every day for 2 weeks or more in a row during the past 12 months that they stopped doing some usual activities.
- **Valid codes used:**  
  - `1` = Yes  
  - `2` = No
- **Recoding rule used in this project:**  
  - `1` -> `sad_binary = 1`  
  - `2` -> `sad_binary = 0`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.
- **Final valid sample size for proportion  analysis:** 13845

In [2]:
behavior_raw = raw["SadOrHopeless"]

behavior_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Code 1 count",
        "Code 2 count",
        "Invalid non-missing values",
        "Final valid sample size"
    ],
    "value": [
        len(behavior_raw),
        int(behavior_raw.isna().sum()),
        int(behavior_raw.notna().sum()),
        int((behavior_raw == 1).sum()),
        int((behavior_raw == 2).sum()),
        int((behavior_raw.notna() & ~behavior_raw.isin([1, 2])).sum()),
        int(behavior_raw.isin([1, 2]).sum())
    ]
})

behavior_check.to_csv(TAB_DIR / "01_behavior_data_check.csv", index=False)
display(behavior_check)

,metric,value
0,Total rows,14041
1,Missing values,196
2,Non-missing values,13845
3,Code 1 count,4153
4,Code 2 count,9692
5,Invalid non-missing values,0
6,Final valid sample size,13845


### 2.2 Continuous Variable for Mean Analysis
- **Variable name:** `HowMuchDoYouWeighWithoutShoesInKG`
- **What the variable measures:** body weight without shoes in kilograms
- **Valid values used:** positive numeric values
- **How missing or invalid values are handled:** removed from the analysis
- **Final valid sample size for mean analysis:** 13062

In [3]:
cont_original = raw[CONT_VAR]
cont_numeric = pd.to_numeric(cont_original, errors="coerce")

missing_count = int(cont_original.isna().sum())
invalid_non_missing_count = int(cont_original.notna().sum() - cont_numeric.notna().sum())
non_positive_count = int((cont_numeric.notna() & (cont_numeric <= 0)).sum())
final_valid_n = int((cont_numeric.notna() & (cont_numeric > 0)).sum())

continuous_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Non-positive values",
        "Invalid non-missing values",
        "Final valid sample size"
    ],
    "value": [
        len(cont_original),
        missing_count,
        int(cont_original.notna().sum()),
        non_positive_count,
        invalid_non_missing_count,
        final_valid_n
    ]
})

continuous_check.to_csv(TAB_DIR / "01_continuous_data_check.csv", index=False)
display(continuous_check)

,metric,value
0,Total rows,14041
1,Missing values,979
2,Non-missing values,13062
3,Non-positive values,0
4,Invalid non-missing values,0
5,Final valid sample size,13062


## 3. Additional EDA:  Variable Definition and Coding

### 3.1 Explanatory variable: CurrentCigaretteUse
- **Variable name:** `CurrentCigaretteUse`
- **What the variable measures:** number of days the student smoked cigarettes during the past 30 days.
- **Valid codes used:**  
  - `1` = 0 days  
  - `2` = 1 or 2 days  
  - `3` = 3 to 5 days  
  - `4` = 6 to 9 days  
  - `5` = 10 to 19 days  
  - `6` = 20 to 29 days  
  - `7` = all 30 days
- **Recoding rule used in this project:**  
  - `1` -> `smoker_binary = 0` (non-smoker in the past 30 days)  
  - `2` to `7` -> `smoker_binary = 1` (reported smoking on at least 1 day in the past 30 days)
- **Additional grouped version used for exploratory EDA:**  
  - `2, 3` -> `Light (1~5 days)`  
  - `4, 5` -> `Moderate (6~19 days)`  
  - `6, 7` -> `Frequent (20~30 days)`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.

In [4]:
current_cig_raw = raw["CurrentCigaretteUse"]

# 原始代碼樣本數
current_cig_counts = (
    current_cig_raw.value_counts(dropna=False)
    .sort_index()
    .rename_axis("code")
    .reset_index(name="count")
)

current_cig_label_map = {
    1: "0 days",
    2: "1 or 2 days",
    3: "3 to 5 days",
    4: "6 to 9 days",
    5: "10 to 19 days",
    6: "20 to 29 days",
    7: "all 30 days"
}

current_cig_counts["code"] = current_cig_counts["code"].astype("object")
current_cig_counts["meaning"] = current_cig_counts["code"].map(current_cig_label_map)
current_cig_counts["code"] = current_cig_counts["code"].where(
    current_cig_counts["code"].notna(), "Missing"
)
current_cig_counts["meaning"] = current_cig_counts["meaning"].where(
    current_cig_counts["meaning"].notna(), "Missing or invalid"
)

current_cig_counts.to_csv(TAB_DIR / "01_current_cigarette_use_code_counts.csv", index=False)
display(current_cig_counts)

# binary recoding 後的樣本數
smoker_binary_counts = (
    processed["smoker_binary"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("smoker_binary")
    .reset_index(name="count")
)

smoker_binary_counts["smoker_binary"] = smoker_binary_counts["smoker_binary"].astype("object")
smoker_binary_counts["smoker_binary"] = smoker_binary_counts["smoker_binary"].replace({
    0.0: "Non-smoker (0)",
    1.0: "Current smoker (1)"
})
smoker_binary_counts["smoker_binary"] = smoker_binary_counts["smoker_binary"].where(
    smoker_binary_counts["smoker_binary"].notna(), "Missing"
)

smoker_binary_counts.to_csv(TAB_DIR / "01_smoker_binary_counts.csv", index=False)
display(smoker_binary_counts)

# 額外 grouped version 的樣本數
smoking_freq_group_counts = (
    processed["smoking_freq_group"]
    .value_counts(dropna=False)
    .rename_axis("smoking_freq_group")
    .reset_index(name="count")
)

smoking_freq_group_counts["smoking_freq_group"] = smoking_freq_group_counts["smoking_freq_group"].astype("object")
smoking_freq_group_counts["smoking_freq_group"] = smoking_freq_group_counts["smoking_freq_group"].where(
    smoking_freq_group_counts["smoking_freq_group"].notna(), "Missing"
)

smoking_freq_group_counts.to_csv(TAB_DIR / "01_smoking_freq_group_counts.csv", index=False)
display(smoking_freq_group_counts)

,code,count,meaning
0,1.0,10734,0 days
1,2.0,753,1 or 2 days
2,3.0,375,3 to 5 days
3,4.0,250,6 to 9 days
4,5.0,295,10 to 19 days
5,6.0,229,20 to 29 days
6,7.0,687,all 30 days
7,Missing,718,Missing or invalid


,smoker_binary,count
0,Non-smoker (0),10734
1,Current smoker (1),2589
2,Missing,718


,smoking_freq_group,count
0,Missing,11452
1,Light (1~5 days),1128
2,Frequent (20~30 days),916
3,Moderate (6~19 days),545


### 3.2 Grouping variable: HowOldAreYou
- **Variable name:** `HowOldAreYou`
- **What the variable measures:** age category of the student.
- **Valid codes used:**  
  - `1` = 12 years old or younger  
  - `2` = 13 years old  
  - `3` = 14 years old  
  - `4` = 15 years old  
  - `5` = 16 years old  
  - `6` = 17 years old  
  - `7` = 18 years old or older
- **Grouping rule used in this project:**  
  - `1, 2, 3` -> `<=14`  
  - `4` -> `15`  
  - `5` -> `16`  
  - `6` -> `17`  
  - `7` -> `18+`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.

**Note on age grouping:**  
In the raw data, the numbers of students coded as `1` (12 or younger) and `2` (13 years old) are very small. To make the subgroup comparison more stable and easier to interpret, this project combines codes `1`, `2`, and `3` into a broader group labeled `<=14`. This keeps the younger students in the analysis while reducing instability from very small subgroup counts.

In [5]:
age_raw = raw["HowOldAreYou"]

# 原始年齡代碼樣本數
age_counts = (
    age_raw.value_counts(dropna=False)
    .sort_index()
    .rename_axis("code")
    .reset_index(name="count")
)

age_code_label_map = {
    1: "<=12",
    2: "13",
    3: "14",
    4: "15",
    5: "16",
    6: "17",
    7: "18+"
}

age_counts["code"] = age_counts["code"].astype("object")
age_counts["meaning"] = age_counts["code"].map(age_code_label_map)
age_counts["code"] = age_counts["code"].where(age_counts["code"].notna(), "Missing")
age_counts["meaning"] = age_counts["meaning"].where(age_counts["meaning"].notna(), "Missing or invalid")

age_counts.to_csv(TAB_DIR / "01_how_old_are_you_code_counts.csv", index=False)
display(age_counts)

# 合併後 age_group 的樣本數（依正確年齡順序排序）
age_group_counts = (
    processed["age_group"]
    .astype("object")
    .where(processed["age_group"].notna(), "Missing")
    .value_counts(dropna=False)
    .rename_axis("age_group")
    .reset_index(name="count")
)

age_group_counts["age_group"] = pd.Categorical(
    age_group_counts["age_group"],
    categories=["<=14", "15", "16", "17", "18+", "Missing"],
    ordered=True
)

age_group_counts = age_group_counts.sort_values("age_group").reset_index(drop=True)

age_group_counts.to_csv(TAB_DIR / "01_age_group_counts.csv", index=False)
display(age_group_counts)

,code,count,meaning
0,1.0,22,<=12
1,2.0,8,13
2,3.0,1361,14
3,4.0,3239,15
4,5.0,3606,16
5,6.0,3575,17
6,7.0,2169,18+
7,Missing,61,Missing or invalid


,age_group,count
0,<=14,1391
1,15,3239
2,16,3606
3,17,3575
4,18+,2169
5,Missing,61


### 3.3 Final sample size used in the exploratory analysis

In [6]:
final_sample = pd.DataFrame({
    "metric": ["Final analysis-ready sample size"],
    "value": [int(processed.dropna(subset=["sad_binary", "smoker_binary", "age_group"]).shape[0])]
})

final_sample.to_csv(TAB_DIR / "01_final_analysis_sample.csv", index=False)
display(final_sample)

,metric,value
0,Final analysis-ready sample size,13120
